# PALMS — PPO Traffic Signal Control
**Palapye Intersection · Single-Agent PPO vs Fixed-Time Baseline**

Run cells top-to-bottom. The Colab setup cell (below) installs SUMO and mounts your Drive automatically.

## 0. Colab Runtime Setup
Installs SUMO, mounts Google Drive, and copies project files into `/content/`.
**Edit `DRIVE_PROJECT` to match your Drive folder name if needed.**

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected — installing dependencies...")
    os.system("apt-get update -qq && apt-get install -y sumo sumo-tools sumo-doc > /dev/null 2>&1")
    os.environ["SUMO_HOME"] = "/usr/share/sumo"
    os.system("pip install -q stable-baselines3[extra] traci gymnasium tensorboard")

    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # ---- EDIT this path to match your Drive folder ----
    DRIVE_PROJECT = "/content/drive/MyDrive/PALMS-Multi-Agent-DRL/SPPO_model"
    # ---------------------------------------------------

    import shutil
    if os.path.exists(DRIVE_PROJECT):
        for fname in os.listdir(DRIVE_PROJECT):
            src = os.path.join(DRIVE_PROJECT, fname)
            if os.path.isfile(src):
                shutil.copy2(src, f"/content/{fname}")
        print("Files copied from Drive:", [f for f in os.listdir("/content") if not f.startswith(".")])
    else:
        print(f"Drive folder not found: {DRIVE_PROJECT}")
        print("Run the manual upload cell below instead.")

    os.chdir("/content")
    print("SUMO_HOME:", os.environ["SUMO_HOME"])
    print("Working dir:", os.getcwd())
else:
    print("Local environment detected.")
    # On Windows SUMO_HOME is usually set already; if not, set it here:
    if "SUMO_HOME" not in os.environ:
        os.environ["SUMO_HOME"] = r"C:\Program Files (x86)\Eclipse\Sumo"
    print("SUMO_HOME:", os.environ.get("SUMO_HOME"))

In [ ]:
# Manual upload fallback — only run this if the Drive cell above failed.
# Upload: env.py, map.sumocfg, map.osm, routes.rou.xml, tls.add.xml,
#         vtypes.add.xml, and any .zip model checkpoints.
if IN_COLAB:
    from google.colab import files
    print("Select all project files to upload...")
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Not in Colab — skip this cell.")

## 1. Imports & Paths

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

# Ensure env.py is importable
PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

SUMO_CFG        = os.path.join(PROJECT_DIR, "map.sumocfg")
CHECKPOINT_DIR  = os.path.join(PROJECT_DIR, "checkpoints")
BEST_MODEL_DIR  = os.path.join(PROJECT_DIR, "best_model")
EVAL_LOG_DIR    = os.path.join(PROJECT_DIR, "eval_logs")
TB_DIR          = os.path.join(PROJECT_DIR, "tb_logs")
MONITOR_DIR     = os.path.join(PROJECT_DIR, "logs")
FINAL_MODEL     = os.path.join(PROJECT_DIR, "ppo_traffic_final.zip")

for d in [CHECKPOINT_DIR, BEST_MODEL_DIR, EVAL_LOG_DIR, TB_DIR, MONITOR_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device       : {DEVICE}")
print(f"Project dir  : {PROJECT_DIR}")
print(f"SUMO config  : {SUMO_CFG}")
print(f"Final model  : {FINAL_MODEL}")
print(f"SUMO_HOME    : {os.environ.get('SUMO_HOME', 'NOT SET')}")

## 2. Environment Inspection

In [ ]:
from env import SumoEnv

env = SumoEnv(sumo_cfg=SUMO_CFG, use_gui=False)

print("Traffic light ID  :", env.tl_id)
print("Action space      :", env.action_space)         # Discrete(4)
print("Observation space :", env.observation_space)    # Box(19,)
print("Green phases      :", env.phase_map)            # [0, 6, 9, 12]
print("Incoming lanes    :", env.incoming_lanes)
print("Outgoing lanes    :", env.outgoing_lanes)
print("Min green time    :", env.min_green_time, "steps")
print("Max episode steps :", env.max_episode_steps)
print("Yellow duration   :", env.yellow_duration, "s")
print("Red (intergreen)  :", env.red_duration, "s")

env.close()

## 3. Train PPO Agent

Skips if `ppo_traffic_final.zip` already exists — delete it to retrain.
Saves a checkpoint every 25 000 steps and keeps the best model in `best_model/`.

In [ ]:
TOTAL_TIMESTEPS = 200_000

if os.path.exists(FINAL_MODEL):
    print(f"Found existing model: {FINAL_MODEL}")
    print("Delete it and re-run to retrain.")
else:
    import time
    run_name = time.strftime("PPO_run_%Y%m%d_%H%M%S")
    run_tb_dir = os.path.join(TB_DIR, run_name)
    os.makedirs(run_tb_dir, exist_ok=True)

    def make_env():
        return Monitor(SumoEnv(sumo_cfg=SUMO_CFG, use_gui=False), MONITOR_DIR)

    train_env = DummyVecEnv([make_env])
    eval_env  = DummyVecEnv([make_env])

    checkpoint_cb = CheckpointCallback(
        save_freq=25_000,
        save_path=CHECKPOINT_DIR,
        name_prefix="ppo_traffic"
    )
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=BEST_MODEL_DIR,
        log_path=EVAL_LOG_DIR,
        eval_freq=10_000,
        deterministic=True,
        render=False
    )

    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        verbose=1,
        tensorboard_log=run_tb_dir,
        device=DEVICE,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=64,
        n_epochs=5,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
    )

    print(f"Training for {TOTAL_TIMESTEPS:,} timesteps on {DEVICE}...")
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=[checkpoint_cb, eval_cb],
        tb_log_name="PPO_traffic_signal"
    )
    model.save(FINAL_MODEL)
    train_env.close()
    eval_env.close()
    print(f"Model saved: {FINAL_MODEL}")
    print(f"TensorBoard logs: {run_tb_dir}")

## 4. Training Reward Curve (from Monitor logs)

In [ ]:
def smooth(y, window=10):
    if len(y) < window:
        return y
    return np.convolve(y, np.ones(window) / window, mode="valid")

monitor_files = [f for f in os.listdir(MONITOR_DIR) if f.endswith(".monitor.csv")]

if not monitor_files:
    print("No monitor logs found — run the training cell first.")
else:
    x, y = ts2xy(load_results(MONITOR_DIR), "timesteps")

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(x, y, alpha=0.25, color="#4472C4", label="Episode reward (raw)")
    y_smooth = smooth(y)
    ax.plot(x[len(x) - len(y_smooth):], y_smooth, color="#4472C4", lw=2,
            label=f"Smoothed (10-ep moving avg)")
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Episode Reward")
    ax.set_title("PPO Training — Episode Reward")
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v/1000)}k"))
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Episodes logged: {len(y)}  |  Final 10-ep avg reward: {np.mean(y[-10:]):.2f}")

## 5. EvalCallback Learning Curve (best-model evaluations)

In [ ]:
eval_npz = os.path.join(EVAL_LOG_DIR, "evaluations.npz")

if not os.path.exists(eval_npz):
    print(f"Not found: {eval_npz} — run training first.")
else:
    data = np.load(eval_npz)
    timesteps  = data["timesteps"]
    results    = data["results"]    # shape (n_evals, n_eval_eps)
    ep_lengths = data["ep_lengths"]

    mean_r = results.mean(axis=1)
    std_r  = results.std(axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    ax.plot(timesteps, mean_r, color="#70AD47", lw=2, label="Mean eval reward")
    ax.fill_between(timesteps, mean_r - std_r, mean_r + std_r,
                    alpha=0.2, color="#70AD47", label="±1 std")
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Mean Episode Reward")
    ax.set_title("Evaluation Reward (EvalCallback)")
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v/1000)}k"))
    ax.legend()
    ax.grid(alpha=0.3)

    ax2 = axes[1]
    ax2.plot(timesteps, ep_lengths.mean(axis=1), color="#ED7D31", lw=2)
    ax2.set_xlabel("Timesteps")
    ax2.set_ylabel("Mean Episode Length (steps)")
    ax2.set_title("Episode Length over Training")
    ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v/1000)}k"))
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f"Best eval mean reward: {mean_r.max():.2f}  at step {timesteps[mean_r.argmax()]:,}")

## 6. Evaluate Trained DRL Agent

In [ ]:
N_EVAL = 5

# Prefer best model saved by EvalCallback; fall back to final
best_model_path = os.path.join(BEST_MODEL_DIR, "best_model.zip")
model_to_load   = best_model_path if os.path.exists(best_model_path) else FINAL_MODEL

if not os.path.exists(model_to_load):
    print(f"No model found at {model_to_load} — train first.")
else:
    print(f"Loading: {model_to_load}")
    model = PPO.load(model_to_load)

    drl_rewards, drl_queues, drl_delays, drl_travel_times = [], [], [], []

    eval_env = SumoEnv(sumo_cfg=SUMO_CFG, use_gui=False)

    for ep in range(N_EVAL):
        obs, _ = eval_env.reset()
        ep_reward = 0.0
        ep_queues, ep_delays, ep_tt = [], [], []
        done = False

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            done = terminated or truncated
            ep_reward += reward
            m = info.get("metrics", {})
            ep_queues.append(m.get("avg_queue", 0.0))
            ep_delays.append(m.get("avg_delay", 0.0))
            ep_tt.append(m.get("avg_travel_time", 0.0))

        drl_rewards.append(ep_reward)
        drl_queues.append(ep_queues)
        drl_delays.append(ep_delays)
        drl_travel_times.append(ep_tt)
        print(f"[DRL] Ep {ep+1}: reward={ep_reward:.1f}  avg_queue={np.mean(ep_queues):.3f}  "
              f"avg_delay={np.mean(ep_delays):.3f}")

    eval_env.close()
    print(f"\nDRL mean reward     : {np.mean(drl_rewards):.2f} ± {np.std(drl_rewards):.2f}")
    print(f"DRL mean queue      : {np.mean([np.mean(q) for q in drl_queues]):.4f}")
    print(f"DRL mean delay      : {np.mean([np.mean(d) for d in drl_delays]):.4f}")

## 7. Fixed-Time Baseline Evaluation

Cycles through all 4 green phases (0 → 6 → 9 → 12) at a fixed interval.
Matches the `evaluate_fixed.py` metrics for a fair comparison.

In [ ]:
PHASE_DURATION = 30  # steps each phase is held before cycling

base_rewards, base_queues, base_delays, base_travel_times = [], [], [], []

base_env = SumoEnv(sumo_cfg=SUMO_CFG, use_gui=False)
# Shorten min_green so fixed-time policy can actually cycle
base_env.min_green_time = 0

phase_cycle = [0, 1, 2, 3]  # action indices → phases [0, 6, 9, 12]

for ep in range(N_EVAL):
    obs, _ = base_env.reset()
    ep_reward = 0.0
    ep_queues, ep_delays, ep_tt = [], [], []
    done = False
    step = 0

    while not done:
        phase_idx = (step // PHASE_DURATION) % len(phase_cycle)
        action = phase_cycle[phase_idx]

        obs, reward, terminated, truncated, info = base_env.step(action)
        done = terminated or truncated
        ep_reward += reward
        m = info.get("metrics", {})
        ep_queues.append(m.get("avg_queue", 0.0))
        ep_delays.append(m.get("avg_delay", 0.0))
        ep_tt.append(m.get("avg_travel_time", 0.0))
        step += 1

    base_rewards.append(ep_reward)
    base_queues.append(ep_queues)
    base_delays.append(ep_delays)
    base_travel_times.append(ep_tt)
    print(f"[Fixed] Ep {ep+1}: reward={ep_reward:.1f}  avg_queue={np.mean(ep_queues):.3f}  "
          f"avg_delay={np.mean(ep_delays):.3f}")

base_env.close()
print(f"\nFixed mean reward   : {np.mean(base_rewards):.2f} ± {np.std(base_rewards):.2f}")
print(f"Fixed mean queue    : {np.mean([np.mean(q) for q in base_queues]):.4f}")
print(f"Fixed mean delay    : {np.mean([np.mean(d) for d in base_delays]):.4f}")

## 8. DRL vs Fixed-Time Comparison

In [ ]:
drl_mean_q   = np.mean([np.mean(q) for q in drl_queues])
base_mean_q  = np.mean([np.mean(q) for q in base_queues])
drl_mean_d   = np.mean([np.mean(d) for d in drl_delays])
base_mean_d  = np.mean([np.mean(d) for d in base_delays])
drl_mean_tt  = np.mean([np.mean(t) for t in drl_travel_times])
base_mean_tt = np.mean([np.mean(t) for t in base_travel_times])

reward_gain  = (np.mean(drl_rewards) - np.mean(base_rewards)) / (abs(np.mean(base_rewards)) + 1e-9) * 100
queue_reduce = (base_mean_q - drl_mean_q) / (base_mean_q + 1e-9) * 100
delay_reduce = (base_mean_d - drl_mean_d) / (base_mean_d + 1e-9) * 100

BLUE, GREEN = "#4472C4", "#70AD47"

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Reward ---
ax = axes[0]
means = [np.mean(base_rewards), np.mean(drl_rewards)]
stds  = [np.std(base_rewards),  np.std(drl_rewards)]
ax.bar(["Fixed-Time", "PPO (DRL)"], means, yerr=stds,
       color=[BLUE, GREEN], capsize=6, width=0.5)
ax.set_ylabel("Mean Episode Reward")
ax.set_title("Episode Reward")
ax.grid(axis="y", alpha=0.3)

# --- Queue length over time (episode 1) ---
ax2 = axes[1]
ax2.plot(base_queues[0], color=BLUE,  alpha=0.8, lw=1.2, label="Fixed-Time")
ax2.plot(drl_queues[0],  color=GREEN, alpha=0.8, lw=1.2, label="PPO (DRL)")
ax2.set_xlabel("Simulation Step")
ax2.set_ylabel("Avg Queue (vehicles)")
ax2.set_title("Queue Length — Episode 1")
ax2.legend()
ax2.grid(alpha=0.3)

# --- Delay over time (episode 1) ---
ax3 = axes[2]
ax3.plot(base_delays[0], color=BLUE,  alpha=0.8, lw=1.2, label="Fixed-Time")
ax3.plot(drl_delays[0],  color=GREEN, alpha=0.8, lw=1.2, label="PPO (DRL)")
ax3.set_xlabel("Simulation Step")
ax3.set_ylabel("Avg Delay (normalised)")
ax3.set_title("Vehicle Delay — Episode 1")
ax3.legend()
ax3.grid(alpha=0.3)

plt.suptitle("PPO DRL vs Fixed-Time Baseline — Palapye Intersection",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# --- Summary table ---
print("="*52)
print(f"{'Metric':<28} {'Fixed-Time':>10} {'PPO DRL':>10}")
print("-"*52)
print(f"{'Mean Episode Reward':<28} {np.mean(base_rewards):>10.2f} {np.mean(drl_rewards):>10.2f}")
print(f"{'Mean Avg Queue (veh)':<28} {base_mean_q:>10.4f} {drl_mean_q:>10.4f}")
print(f"{'Mean Avg Delay (norm.)':<28} {base_mean_d:>10.4f} {drl_mean_d:>10.4f}")
print(f"{'Mean Travel Time (s)':<28} {base_mean_tt:>10.2f} {drl_mean_tt:>10.2f}")
print("-"*52)
print(f"Reward improvement  : {reward_gain:+.1f}%")
print(f"Queue reduction     : {queue_reduce:+.1f}%")
print(f"Delay reduction     : {delay_reduce:+.1f}%")
print("="*52)

## 9. Checkpoint Analysis (optional)

Loads each saved checkpoint and runs a quick 1-episode eval to track how the agent improved during training.

In [ ]:
import glob, re

ckpts = sorted(
    glob.glob(os.path.join(CHECKPOINT_DIR, "ppo_traffic_*_steps.zip")),
    key=lambda p: int(re.search(r"_(\d+)_steps", p).group(1))
)

if not ckpts:
    print("No checkpoints found — run training first.")
else:
    print(f"Found {len(ckpts)} checkpoints.")
    ckpt_steps, ckpt_rewards, ckpt_queues = [], [], []
    ckpt_env = SumoEnv(sumo_cfg=SUMO_CFG, use_gui=False)

    for ckpt in ckpts:
        step_num = int(re.search(r"_(\d+)_steps", ckpt).group(1))
        m = PPO.load(ckpt)
        obs, _ = ckpt_env.reset()
        r_sum, q_list, done = 0.0, [], False
        while not done:
            a, _ = m.predict(obs, deterministic=True)
            obs, r, terminated, truncated, info = ckpt_env.step(a)
            done = terminated or truncated
            r_sum += r
            q_list.append(info.get("metrics", {}).get("avg_queue", 0.0))
        ckpt_steps.append(step_num)
        ckpt_rewards.append(r_sum)
        ckpt_queues.append(np.mean(q_list))
        print(f"  Step {step_num:>7,}: reward={r_sum:.1f}  avg_queue={np.mean(q_list):.4f}")

    ckpt_env.close()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(ckpt_steps, ckpt_rewards, marker="o", color=GREEN)
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Episode Reward")
    axes[0].set_title("Reward vs Checkpoint")
    axes[0].xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v/1000)}k"))
    axes[0].grid(alpha=0.3)

    axes[1].plot(ckpt_steps, ckpt_queues, marker="o", color=BLUE)
    axes[1].set_xlabel("Training Step")
    axes[1].set_ylabel("Avg Queue (vehicles)")
    axes[1].set_title("Queue Length vs Checkpoint")
    axes[1].xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v/1000)}k"))
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, "checkpoint_progress.png"), dpi=150)
    plt.show()